# Random Forest Model — UCI Cannabis Risk Dataset

This notebook builds Random Forest as the primary model, tunes hyperparameters, compares it against Logistic Regression (baseline), MLP, and KNN, assigns continuous risk tiers, and saves the final model as a `.joblib` file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json
import os

from sklearn.ensemble        import RandomForestClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.neural_network  import MLPClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (classification_report, confusion_matrix,
                                     roc_auc_score, roc_curve,
                                     accuracy_score, precision_score,
                                     recall_score, f1_score)
from sklearn.utils.class_weight import compute_sample_weight

In [ ]:
df_train = pd.read_csv('uci_cannabis_train.csv')
df_test  = pd.read_csv('uci_cannabis_test.csv')

display(df_train.head())
display(df_test.head())

In [ ]:
# Separating features and target variable
FEATURES = ['Age','Gender','Education','Country','Ethnicity',
            'Nscore','Escore','Oscore','Ascore','Cscore','Impulsive','SS']

X_train = df_train[FEATURES]
y_train = df_train['cannabis_risk']

X_test  = df_test[FEATURES]
y_test  = df_test['cannabis_risk']

feature_cols = X_train.columns.tolist()
len(X_train), len(X_test), feature_cols

In [ ]:
# Separating features and target (for validation — uses train split internally)
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

## Random Forest — No Scaling Required

- Random Forest is a tree-based ensemble model — it splits on feature thresholds, not distances or gradients.
- Unlike Logistic Regression, MLP, and KNN, it does **not** require feature scaling.
- We still scale separately for the comparison models (LR, MLP, KNN) later.

In [ ]:
# Baseline Random Forest
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_baseline.fit(X_tr, y_tr)

y_pred = rf_baseline.predict(X_val)

print('Baseline Random Forest (validation set):')
print('Accuracy: ', accuracy_score(y_val, y_pred))
print('Precision:', precision_score(y_val, y_pred))
print('Recall:   ', recall_score(y_val, y_pred))
print('F1 Score: ', f1_score(y_val, y_pred))

In [ ]:
# Tuning n_estimators — number of trees in the forest
# More trees = more stable predictions, but diminishing returns beyond ~200
n_estimators_range = [50, 100, 200, 300, 500]
results_n = []

for n in n_estimators_range:
    rf = RandomForestClassifier(
        n_estimators=n,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_val)
    results_n.append({
        'n_estimators': n,
        'Accuracy' : accuracy_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
        'Recall'   : recall_score(y_val, y_pred),
        'F1 Score' : f1_score(y_val, y_pred)
    })

pd.DataFrame(results_n)

- n_estimators=200 gives the best balance of recall and F1 with stable performance.
- Beyond 200 trees there are diminishing returns — performance plateaus while training time increases.

In [ ]:
# Tuning max_depth — maximum depth of each tree
# None = trees grow until all leaves are pure (can overfit)
# Shallow trees underfit; deep trees capture more complex patterns
depth_range = [None, 5, 10, 15, 20]
results_d = []

for d in depth_range:
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=d,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_val)
    results_d.append({
        'max_depth': str(d),
        'Accuracy' : accuracy_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
        'Recall'   : recall_score(y_val, y_pred),
        'F1 Score' : f1_score(y_val, y_pred)
    })

pd.DataFrame(results_d)

- max_depth=None (fully grown trees) gives the best recall and F1 on this dataset.
- Restricting depth hurts recall — important since our goal is to catch as many at-risk students as possible.

In [ ]:
# Threshold tuning — sweep classification threshold to maximise recall
# Default threshold is 0.5: predict At Risk if prob >= 0.5
# Lowering it catches more at-risk students (higher recall) at cost of precision
#
# In a harm-prevention context, a false negative (missed at-risk student)
# is far more costly than a false positive (unnecessary flag).

rf_tuned = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_tuned.fit(X_tr, y_tr)
y_proba_val = rf_tuned.predict_proba(X_val)[:, 1]

thresholds = [0.5, 0.45, 0.40, 0.35, 0.30]
results_thr = []

for thr in thresholds:
    y_pred_thr = (y_proba_val >= thr).astype(int)
    results_thr.append({
        'Threshold': thr,
        'Accuracy' : accuracy_score(y_val, y_pred_thr),
        'Precision': precision_score(y_val, y_pred_thr),
        'Recall'   : recall_score(y_val, y_pred_thr),
        'F1'       : f1_score(y_val, y_pred_thr)
    })

pd.DataFrame(results_thr)

- Threshold 0.45 gives the best F1 (0.8636) while achieving 0.913 recall — catching over 91% of at-risk students.
- Threshold 0.40 pushes recall to 0.933 with an acceptable drop in precision.
- **Threshold 0.45 selected** as the optimal balance for this screening tool.

In [ ]:
# Refit on full training set with chosen hyperparameters
# n_estimators=200, max_depth=None, threshold=0.45

rf_final = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_final.fit(X_train, y_train)

threshold = 0.45
y_test_proba = rf_final.predict_proba(X_test)[:, 1]
y_test_pred  = (y_test_proba >= threshold).astype(int)

print(f'=== Random Forest (threshold={threshold}, test set) ===')
print(classification_report(y_test, y_test_pred,
      target_names=['Non-user', 'At Risk'], digits=3))
print(f'ROC-AUC: {roc_auc_score(y_test, y_test_proba):.4f}')
print(confusion_matrix(y_test, y_test_pred))

In [ ]:
# Feature importance — built-in to Random Forest
# Shows which personality/demographic features drive the predictions most
FEATURE_LABELS = ['Age','Gender','Education','Country','Ethnicity',
                  'Neuroticism','Extraversion','Openness',
                  'Agreeableness','Conscientiousness','Impulsivity','Sensation-Seeking']

importances = pd.Series(rf_final.feature_importances_, index=FEATURE_LABELS).sort_values()

importances.plot(kind='barh', figsize=(9, 5), color='steelblue', alpha=0.85)
plt.xlabel('Feature Importance (Gini)')
plt.title('Random Forest — Feature Importance')
plt.tight_layout()
plt.show()

print(importances.sort_values(ascending=False).round(4))

- **Country**, **Age**, **Openness**, and **Sensation-Seeking** are the strongest predictors of cannabis risk.
- This aligns with published literature — Sensation-Seeking in particular is consistently cited as a top predictor of cannabis use (Fehrman et al., 2015).
- Feature importance is a key advantage of Random Forest over MLP — we can explain *why* the model makes each prediction.

In [ ]:
# Full model comparison — LR vs RF vs MLP vs KNN
# Scaling needed for LR, MLP, KNN (not RF)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

log_reg = LogisticRegression(
    penalty='l2', C=1.0, class_weight='balanced',
    solver='lbfgs', max_iter=1000, random_state=42
)

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
    max_iter=300, early_stopping=True, validation_fraction=0.1, random_state=42
)

knn = KNeighborsClassifier(n_neighbors=5, weights='distance', metric='euclidean')

log_reg.fit(X_train_s, y_train)
mlp.fit(X_train_s, y_train, sample_weight=sample_weights)
knn.fit(X_train_s, y_train)

comparison = []
for name, model, X_t in [
    ('Logistic Regression', log_reg, X_test_s),
    ('Random Forest',       rf_final, X_test),   # RF uses unscaled data
    ('MLP',                 mlp,     X_test_s),
    ('KNN',                 knn,     X_test_s),
]:
    proba  = model.predict_proba(X_t)[:, 1]
    y_pred = (proba >= threshold).astype(int)
    comparison.append({
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall'   : round(recall_score(y_test, y_pred), 4),
        'F1 Score' : round(f1_score(y_test, y_pred), 4),
        'ROC-AUC'  : round(roc_auc_score(y_test, proba), 4),
    })

pd.DataFrame(comparison)

- **Random Forest achieves the highest recall and F1** across all four models.
- For this harm-prevention screening tool, recall is the most important metric — missing an at-risk student is more costly than a false alarm.
- Random Forest also provides built-in feature importance, making results interpretable and academically justifiable.
- Logistic Regression remains a strong, interpretable baseline.
- MLP and KNN are competitive but offer no advantage over Random Forest on this dataset.

In [ ]:
# ROC curve comparison — all four models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
ax = axes[0]
for name, model, X_t, colour in [
    ('Logistic Regression', log_reg, X_test_s, '#8B5CF6'),
    ('Random Forest',       rf_final, X_test,  '#22C55E'),
    ('MLP',                 mlp,     X_test_s, '#3B82F6'),
    ('KNN',                 knn,     X_test_s, '#F97316'),
]:
    proba = model.predict_proba(X_t)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=colour, linewidth=2)

ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Feature importance bar chart
ax2 = axes[1]
importances.sort_values().plot(kind='barh', ax=ax2, color='steelblue', alpha=0.85)
ax2.set_xlabel('Feature Importance (Gini)')
ax2.set_title('Random Forest — Feature Importance')

plt.tight_layout()
plt.savefig('rf_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved: rf_model_comparison.png')

In [ ]:
# Continuous risk tier assignment
# Probability mapped to actionable tiers for the screening tool
def assign_risk_tier(prob):
    if   prob < 0.30: return 'Low Risk'
    elif prob < 0.55: return 'Moderate Risk'
    elif prob < 0.75: return 'High Risk'
    else:             return 'Very High Risk'

rf_tiers = [assign_risk_tier(p) for p in y_test_proba]

tier_counts = pd.Series(rf_tiers).value_counts().reindex(
    ['Low Risk','Moderate Risk','High Risk','Very High Risk'], fill_value=0)

print('Random Forest — Risk Tier Distribution (Test Set):')
for tier, count in tier_counts.items():
    print(f'  {tier:<18}: {count:>3}  ({count/len(rf_tiers)*100:.1f}%)')

In [ ]:
# Save final model and metadata
os.makedirs('ml_artifacts', exist_ok=True)

joblib.dump(rf_final, 'ml_artifacts/rf_cannabis_model.joblib')

metadata = {
    'feature_cols'  : feature_cols,
    'risk_threshold': float(threshold),
    'model_type'    : 'RandomForestClassifier',
    'n_estimators'  : 200,
    'max_depth'     : None,
    'target'        : 'cannabis_risk',
    'dataset'       : 'UCI Drug Consumption (Quantified)',
    'auc_test'      : round(roc_auc_score(y_test, y_test_proba), 4),
}

with open('ml_artifacts/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved: ml_artifacts/rf_cannabis_model.joblib')
print('Saved: ml_artifacts/model_metadata.json')

In [ ]:
# Reload and verify — mirrors what the Django website will do
loaded_model = joblib.load('ml_artifacts/rf_cannabis_model.joblib')

# Example: young male, high sensation-seeking, high openness
example = {
    'Age': -0.95197, 'Gender': -0.48246, 'Education': 1.98437,
    'Country': 0.96082, 'Ethnicity': -0.31685, 'Nscore': 0.84580,
    'Escore': -0.30172, 'Oscore': 1.43533, 'Ascore': -0.80615,
    'Cscore': -0.01928, 'Impulsive': 0.59042, 'SS': 1.30612
}

x = np.array([[example[f] for f in feature_cols]])
prob = loaded_model.predict_proba(x)[0, 1]

print(f'Risk Probability : {prob:.4f}')
print(f'Risk Tier        : {assign_risk_tier(prob)}')
print(f'At Risk Flag     : {"YES" if prob >= threshold else "NO"}')